In [1]:
import csv

with open("processed-actual-outages.csv") as file:
    csv_reader = csv.DictReader(file)
    actual_data = list(csv_reader)

In [2]:
from datetime import datetime

from tqdm import tqdm

date_keys = ["OutDatetime", "MinTimeStamp", "MaxTimeStamp"]

for row in tqdm(actual_data):
    row["PTID"] = int(row["PTID"])
    row["Voltage"] = int(row["Voltage"])
    for key in date_keys:
        row[key] = datetime.strptime(row[key], "%Y-%m-%d %H:%M:%S")

actual_data[0]

100%|██████████| 74063/74063 [00:01<00:00, 64033.41it/s]


{'PTID': 26053,
 'Name': 'MOUNTAIN-SWANROAD_115_104-3',
 'OutDatetime': datetime.datetime(2005, 1, 1, 0, 0),
 'MinTimeStamp': datetime.datetime(2012, 9, 11, 11, 52, 17),
 'MaxTimeStamp': datetime.datetime(2013, 8, 8, 8, 37, 17),
 'Voltage': 115,
 'FirstBus': 'MOUNTAIN',
 'SecondBus': 'SWANROAD',
 'OutageType': 'Planned'}

In [3]:
import igraph

our_bus_set = set(
    [row[key] for row in actual_data for key in ["FirstBus", "SecondBus"]]
)
our_bus_list = sorted(our_bus_set)

print(f"num vertices: {len(our_bus_set)}")

bus_name_to_index = {bus_name: i for i, bus_name in enumerate(our_bus_list)}
index_to_bus_name = {i: bus_name for i, bus_name in enumerate(our_bus_list)}
n = len(bus_name_to_index)
g = igraph.Graph(n=len(bus_name_to_index), directed=False)

edge_set = set()
for datum in actual_data:
    b1, b2 = datum["FirstBus"], datum["SecondBus"]
    b1_index, b2_index = bus_name_to_index[b1], bus_name_to_index[b2]
    edge_set.add((b1_index, b2_index))
g.add_edges(list(edge_set))

print(g.is_connected())

num vertices: 1376
True


In [4]:
# export to multiple graph formats (igraph supports these)
g.write_pickle("res/outage_graph.pkl")

In [5]:
import json
from pathlib import Path

bus_name_to_index_path = Path("res/bus_name_to_index.json")
index_to_bus_name_path = Path("res/index_to_bus_name.json")

bus_name_to_index_path.write_text(json.dumps(bus_name_to_index, indent=4))
index_to_bus_name_path.write_text(json.dumps(index_to_bus_name, indent=4))

31916

In [6]:
import re


def preprocess_station_name(name: str) -> str:
    """
    Normalize compact station names such as:
    'E__SHORE' -> 'East Shore'
    'SCOV RCK' -> 'Scov Rock'
    'COMMACK_' -> 'Commack'

    Rules:
    - replace underscores with spaces
    - collapse repeated spaces
    - strip leading/trailing spaces
    - expand standalone direction letters:
        N -> North, S -> South, E -> East, W -> West
    - title-case the result
    """
    if not isinstance(name, str):
        raise TypeError("name must be a string")

    # Replace underscores with spaces
    s = name.replace("_", " ")

    # Collapse multiple spaces and strip
    s = re.sub(r"\s+", " ", s).strip()

    # Expand standalone cardinal directions
    direction_map = {
        "N.": "North",
        "S.": "South",
        "E.": "East",
        "W.": "West",
    }

    words = s.split()
    words = [direction_map.get(word, word) for word in words]

    # Title-case each word
    s = " ".join(words).title()

    return s

# find lat-long (Run once and manually edit)

In [7]:
from pathlib import Path
import re
import zipfile

import geopandas as gpd
import pandas as pd
from rapidfuzz import process, fuzz


ZIP_PATH = Path("geo/new-york-260311-free.shp.zip")


def normalize_for_match(s: str) -> str:
    """
    Stronger normalization for fuzzy matching.
    """
    s = s.lower().strip()
    s = s.replace("_", " ")
    s = re.sub(r"\b(north|south|east|west)\b", "", s)
    s = re.sub(r"\b(n|s|e|w)\b", "", s)
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def list_zip_shapefiles(zip_path: Path):
    with zipfile.ZipFile(zip_path) as zf:
        return [n for n in zf.namelist() if n.endswith(".shp")]


# 1) Inspect zip contents
shp_files = list_zip_shapefiles(ZIP_PATH)
print("Shapefiles inside zip:")
for shp in shp_files:
    print(" -", shp)


# 2) Load candidate layers
#
# For Geofabrik/OpenStreetMap exports, these are often the most useful
# for place-name matching:
#   - gis_osm_places_free_1.shp
#   - gis_osm_pois_free_1.shp
#
# Adjust these names after you inspect `shp_files`.
candidate_layers = [
    shp for shp in shp_files if "places" in shp.lower() or "pois" in shp.lower()
]

print("\nCandidate layers:")
for shp in candidate_layers:
    print(" -", shp)

gdfs = []
for shp in candidate_layers:
    gdf = gpd.read_file(f"zip://{ZIP_PATH}!{shp}")
    gdf["__source_layer"] = shp
    gdfs.append(gdf)

if not gdfs:
    raise RuntimeError(
        "No candidate shapefiles found. Inspect the zip contents manually."
    )

candidates = pd.concat(gdfs, ignore_index=True)

print("\nColumns:")
print(candidates.columns.tolist())


# 3) Keep rows that have a name and point geometry
if "name" not in candidates.columns:
    raise RuntimeError("No 'name' column found in candidate layers.")

candidates = candidates[candidates["name"].notna()].copy()
candidates = candidates[candidates.geometry.notna()].copy()

# Reproject to lat/lon if needed
if candidates.crs is not None and candidates.crs.to_epsg() != 4326:
    candidates = candidates.to_crs(epsg=4326)

# Extract representative lat/lon
# If geometry is not a point, centroid gives a usable approximate coordinate.
candidates["lon"] = candidates.geometry.centroid.x
candidates["lat"] = candidates.geometry.centroid.y

# Build normalized match keys
candidates["name_clean"] = candidates["name"].astype(str).map(normalize_for_match)

# Drop empty names after normalization
candidates = candidates[candidates["name_clean"] != ""].copy()

# Deduplicate candidate names while preserving first occurrence
candidate_name_map = candidates.drop_duplicates(subset=["name_clean"]).set_index(
    "name_clean"
)

candidate_keys = candidate_name_map.index.tolist()


# 4) Match your station names to shapefile names
rows = []
for bus_name in tqdm(our_bus_list):
    query = normalize_for_match(preprocess_station_name(bus_name))

    if not query:
        rows.append(
            {
                "raw_name": bus_name,
                "clean_name": query,
                "matched_name": None,
                "score": None,
                "lat": None,
                "lon": None,
                "source_layer": None,
            }
        )
        continue

    best = process.extractOne(
        query,
        candidate_keys,
        scorer=fuzz.WRatio,
    )

    if best is None:
        rows.append(
            {
                "raw_name": bus_name,
                "clean_name": query,
                "matched_name": None,
                "score": None,
                "lat": None,
                "lon": None,
                "source_layer": None,
            }
        )
        continue

    matched_key, score, _ = best
    rec = candidate_name_map.loc[matched_key]

    rows.append(
        {
            "raw_name": bus_name,
            "clean_name": query,
            "matched_name": rec["name"],
            "score": score,
            "lat": rec["lat"],
            "lon": rec["lon"],
            "source_layer": rec["__source_layer"],
        }
    )

results = pd.DataFrame(rows)

print("\nMatch results:")
print(results.sort_values(["score", "raw_name"], ascending=[False, True]))

# Save for manual review
results.to_csv("station_matches.csv", index=False)
print("\nSaved: station_matches.csv")

Shapefiles inside zip:
 - gis_osm_buildings_a_free_1.shp
 - gis_osm_landuse_a_free_1.shp
 - gis_osm_natural_a_free_1.shp
 - gis_osm_natural_free_1.shp
 - gis_osm_places_a_free_1.shp
 - gis_osm_places_free_1.shp
 - gis_osm_pofw_a_free_1.shp
 - gis_osm_pofw_free_1.shp
 - gis_osm_pois_a_free_1.shp
 - gis_osm_pois_free_1.shp
 - gis_osm_railways_free_1.shp
 - gis_osm_roads_free_1.shp
 - gis_osm_traffic_a_free_1.shp
 - gis_osm_traffic_free_1.shp
 - gis_osm_transport_a_free_1.shp
 - gis_osm_transport_free_1.shp
 - gis_osm_water_a_free_1.shp
 - gis_osm_waterways_free_1.shp

Candidate layers:
 - gis_osm_places_a_free_1.shp
 - gis_osm_places_free_1.shp
 - gis_osm_pois_a_free_1.shp
 - gis_osm_pois_free_1.shp

Columns:
['osm_id', 'code', 'fclass', 'population', 'name', 'geometry', '__source_layer']


/tmp/ipykernel_35906/1101955290.py:84: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  candidates["lon"] = candidates.geometry.centroid.x
/tmp/ipykernel_35906/1101955290.py:85: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  candidates["lat"] = candidates.geometry.centroid.y
100%|██████████| 1376/1376 [00:48<00:00, 28.37it/s]


Match results:
      raw_name clean_name                    matched_name  score        lat  \
12    ADAMS___      adams                     North Adams  100.0  43.886173   
14    AFTON___      afton                     North Afton  100.0  42.267857   
23    ALBANY__     albany                     West Albany  100.0  42.683134   
35    ALTAMONT   altamont                        Altamont  100.0  42.700632   
36    AMAWALK_    amawalk                         Amawalk  100.0  41.288427   
...        ...        ...                             ...    ...        ...   
1372  YOUNG214   young214                               1   90.0  40.661206   
1373  ZIMMERM1   zimmerm1                               1   90.0  40.661206   
1374  ZIMMERM2   zimmerm2                               2   90.0  40.662021   
1375  __BELLOW     bellow                               l   90.0  40.770135   
849   NEWP_PS_    newp ps  PS 9 Sarah Smith Garnet School   85.5  40.678550   

            lon               sourc

In [8]:
unresolved = results[
    results["lat"].isna() | results["lon"].isna() | (results["score"].fillna(0) < 85)
].copy()

# all are resolved but they're not correctly matched